In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import interpolate
import tqdm
import re
import os
import glob
import spectres
from astropy import units as u

%matplotlib inline

In [56]:
SPS_HOME = os.path.abspath(os.path.join(os.getcwd(), '..'))
# SPS_HOME = os.getenv('SPS_HOME')
# SPS_HOME = SPS_HOME.replace('fsps', 'fsps_dev')  # -> I call my development folder for fsps 'fsps_dev' to keep it separate from my working fsps installation

# choose one metallicity value for this run
logzi = np.log10(2.)
# Convert from log(Z/Zsun) to Z/Zsun 
zfrac_s = round(10**logzi, 1)
if zfrac_s >= 1 or zfrac_s == 0:
    zfrac_s = int(zfrac_s)

print(SPS_HOME)
print(zfrac_s)

/Users/mreefe/Dropbox/Astrophysics/fsps_dev
2


In [57]:
# Define the teff, logg, and logz arrays that cover the grid of UVBLUE models

teff_1 = np.arange(15000, 30000, 1000)
teff_2 = np.arange(30000, 55000+2500, 2500)
teff = np.concatenate((teff_1, teff_2))
logt = np.log10(teff)

logg = np.arange(1.75, 4.75+0.25, 0.25)

zfrac = np.array([0., 0.1, 0.2, 0.5, 1., 2.])
logz = np.round(np.log10(zfrac), 1)

print(logt)
print(logg)
print(logz)

[4.17609126 4.20411998 4.23044892 4.25527251 4.2787536  4.30103
 4.32221929 4.34242268 4.36172784 4.38021124 4.39794001 4.41497335
 4.43136376 4.44715803 4.462398   4.47712125 4.51188336 4.54406804
 4.57403127 4.60205999 4.62838893 4.65321251 4.67669361 4.69897
 4.7201593  4.74036269]
[1.75 2.   2.25 2.5  2.75 3.   3.25 3.5  3.75 4.   4.25 4.5  4.75]
[-inf -1.  -0.7 -0.3  0.   0.3]


/var/folders/_d/811k0zw90qn5rglh8sn17ryr0000gn/T/ipykernel_76225/307770014.py:11: RuntimeWarning: divide by zero encountered in log10
  logz = np.round(np.log10(zfrac), 1)


In [58]:
# Define the wavelength grid that we will resample onto
# -> even logarithmic spacing by ~5% of the current wavelength
w_1 = 90. * 1.05**np.arange(0, int(np.log(800/90)/np.log(1.05))) 
# -> finer sampling in the FUV;
w_2 = np.arange(800., 1800., 0.2)
w_3 = np.arange(1800., 9000., 20.)
# -> spacing by ~5% of the current wavelength
w_4 = 9000. * 1.05**np.arange(0, int(np.log(9.99e6/9000)/np.log(1.05))) 

wavelength = np.concatenate((w_1, w_2, w_3, w_4))
print('w_1 = ', len(w_1))
print('w_2 = ', len(w_2))
print('w_3 = ', len(w_3))
print('w_4 = ', len(w_4))
print('length = ', len(wavelength))
print(wavelength)

w_1 =  44
w_2 =  5000
w_3 =  360
w_4 =  143
length =  5547
[9.00000000e+01 9.45000000e+01 9.92250000e+01 ... 8.33190634e+06
 8.74850165e+06 9.18592674e+06]


In [59]:
# wavelength must be in angstroms!
def airtovac(wavelength):
    # see: https://www.astro.uu.se/valdwiki/Air-to-vacuum%20conversion
    s = 1e4 / wavelength
    n = 1 + 0.00008336624212083 + 0.02408926869968 / (130.1065924522 - s**2) + 0.0001599740894897 / (38.92568793293 - s**2)
    # do not alter wavelengths below 2000 angstroms 
    wh = np.where(wavelength < 2000.)[0]
    n[wh] = 1.0
    return wavelength * n

In [60]:

# allocate a buffer for all of the spectra at one metallicity
library_in = np.zeros((len(wavelength), len(logt), len(logg)))

folder = f'/Users/mreefe/Dropbox/Astrophysics/stellar_templates/TLUSTY_OSTAR_BSTAR/{zfrac_s}Zsun/'
files = glob.glob(os.path.join(folder, '*.dat.txt'))

for fpath in tqdm.tqdm(files):

    # parse the file name to get the temp, logg, and logz
    fname = os.path.basename(fpath)
    m = re.search(r'^t([0-9]+)\.g(\d\.\d+)z(\d[\.]?[\d+]?)\.dat\.txt$', fname)
    teff_v = int(m.group(1))
    logg_v = float(m.group(2))
    logz_v = np.round(np.log10(float(m.group(3))), 1)

    # find the indices corresponding to these values in the array
    logt_i = np.where(teff == teff_v)[0][0]
    logg_i = np.where(logg == logg_v)[0][0]

    # print(f'Teff = {teff_v:.0f} (index = {logt_i:.0f})')
    # print(f'logg = {logg_v:.1f} (index = {logg_i:.0f})')
    # print(f'logZ = {logz_v:.1f}')

    # read in the text file
    wave_i, flux_i = np.loadtxt(fpath, unpack=True, skiprows=7)

    # perform the flux-conserving resampling onto the output wavelength grid
    # flux_o = spectres.spectres(wavelength, wave_i, flux_i, fill=0., verbose=False)
    flux_o = np.interp(wavelength, wave_i, flux_i, left=0., right=0.)

    # insert it into the 3D array
    library_in[:, logt_i, logg_i] = flux_o

    # # plot the new and old spectrum to compare them
    # fig, ax = plt.subplots()
    # ax.plot(wave_i, flux_i)
    # ax.plot(wavelength, flux_o)
    # ax.set_xscale('log')
    # ax.set_yscale('log')
    # ax.set_xlabel('Wavelength (angstroms)')
    # ax.set_ylabel('Flambda')
    # # ax.set_xlim(900, 1800)
    # # ax.set_ylim(flux_i[(wave_i > 900) & (wave_i < 1800)].min()*0.98, flux_i[(wave_i > 900) & (wave_i < 1800)].max()*1.02)
    # plt.show()
    # plt.close()


100%|██████████| 216/216 [00:00<00:00, 231.22it/s]


In [61]:
# Convert the units
c_ang = 299792458e10

for i in range(len(logt)):
    for j in range(len(logg)):
        library_in[:,i,j] *= wavelength**2 / c_ang    # <= convert to erg/s/cm2/Hz
        library_in[:,i,j] *= 1/(4*np.pi)              # <= convert to Harvard flux
        # note: insofar as I can tell, FSPS stores its stellar libraries in flux moment or "Harvard flux" units, 
        #       which are off from physical flux units by a factor of 4pi.  See page 244-245 in 
        #       https://ads.harvard.edu/books/1989fsa..book/AbookC09.pdf for more info.
        #       Also see line 199 in getspec.f90 which does the conversion from these units into Lsun/Hz.


In [62]:
# Normalize to unity, to match the WMBasic grids
for i in range(len(logt)):
    for j in range(len(logg)):
        norm = np.trapz(library_in[:,i,j]*c_ang/wavelength**2, wavelength)
        if norm > 0:
            library_in[:,i,j] /= norm

In [63]:
# # Read in WMBasic templates for comparison
# wmb_logt = np.loadtxt(os.path.join(SPS_HOME, 'SPECTRA/Hot_spectra/WMBASIC.teff'))
# wmb_logg = np.array([3.5, 4., 4.5])
# wmb = np.loadtxt(os.path.join(SPS_HOME, 'SPECTRA/Hot_spectra/WMBASIC_z0.0140.spec'))
# w_wmb = wmb[:,0]
# wmb = wmb[:,1:]
# wmb = wmb.reshape(wmb.shape[0], len(wmb_logg), len(wmb_logt))

# template_folder = os.path.join(SPS_HOME, 'SPECTRA/Hot_spectra/TLUSTY_OB/template_plots')
# if not os.path.exists(template_folder):
#     os.makedirs(template_folder)

# for j in range(len(wmb_logg)):
#     for i in range(len(wmb_logt)):
#         jj = np.nanargmin(np.abs(wmb_logg[j] - logg))
#         ii = np.nanargmin(np.abs(wmb_logt[i] - logt))
#         fig, ax = plt.subplots()
#         ax.plot(w_wmb, wmb[:,j,i], label='WMBASIC')
#         ax.plot(wavelength, library_in[:,ii,jj], label='TLUSTY')
#         ax.set_xscale('log')
#         ax.set_yscale('log')
#         ax.set_xlabel(r'Wavelength ($\mathring{\rm A}}$)')
#         ax.set_ylabel(r'Flux moment (erg s$^{-1}$ cm$^{-2}$ Hz$^{-1}$)')
#         # ax.set_xlim(900, 1800)
#         # ax.set_ylim(1e-22, 1e-14)
#         ax.set_title(f'logg={wmb_logg[j]}, teff={10**wmb_logt[i]}')
#         ax.legend()
#         plt.savefig(os.path.join(template_folder, f'{j}_{i}.pdf'), dpi=300, bbox_inches='tight')
#         plt.close()

In [64]:
library_in.shape

(5547, 26, 13)

In [65]:
# Save as a text file with the same format as the format 
zsun = 0.0134
z = zfrac_s * zsun
# need to flatten the array to 2D matching the shape of the WM-Basic arrays FSPS uses
# (flatten logt and logg axis, should be ordered as [t1_g1 t2_g1 t3_g1 ... t1_g2 t2_g2 t3_g2 ...])
library_out = library_in.reshape(library_in.shape[0], len(logg)*len(logt))
# append the wavelength column
library_out = np.concatenate((wavelength.reshape(len(wavelength), 1), library_out), axis=1)

np.savetxt(os.path.join(SPS_HOME, f'SPECTRA/Hot_spectra/TLUSTY_OB/TLUSTYOBz{z:.4f}.spec'), library_out, fmt='%.4e', delimiter=' ')

In [66]:
# save other ancillary files
np.savetxt(os.path.join(SPS_HOME, 'SPECTRA/Hot_spectra/TLUSTY_OB/TLUSTYOB.teff'), logt, fmt='%13.5f')
np.savetxt(os.path.join(SPS_HOME, 'SPECTRA/Hot_spectra/TLUSTY_OB/TLUSTYOB.logg'), logg, fmt='%13.5f')
np.savetxt(os.path.join(SPS_HOME, 'SPECTRA/Hot_spectra/TLUSTY_OB/TLUSTYOB_zlegend.dat'), zfrac*zsun, fmt='%7.4f')